In [7]:
import os
import time
import random
import pyproj
 
# Fix PROJ database conflict caused by PostgreSQL / PostGIS installation
os.environ["PROJ_LIB"] = pyproj.datadir.get_data_dir()
os.environ["PROJ_DATA"] = pyproj.datadir.get_data_dir()
 
# Let GDAL itself auto-retry transient HTTP errors (incl. 429s) when it
# streams pixel windows from the COG — this alone fixes most rate-limit
# errors that happen *during* raster reads rather than during the search.
os.environ["GDAL_HTTP_MAX_RETRY"] = "6"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "3"
 
import numpy as np
import pandas as pd
import planetary_computer
import pystac_client
import rioxarray
from pyproj import Transformer
from requests.exceptions import HTTPError, RequestException
from tqdm import tqdm
 
# --------------------------------------------------------------- settings
OUTPUT_CSV = "cairo_neighborhoods_green.csv"
BUFFER_METERS = 1000
CHECKPOINT_EVERY = 50  # rows between disk saves
 
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
 
_raster_cache = {}  # item.id -> opened rioxarray DataArray (native CRS)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32636", always_xy=True)
 
 
def _retry(fn, *args, max_attempts=6, base_delay=2, **kwargs):
    """Exponential backoff with jitter for network calls."""
    for attempt in range(max_attempts):
        try:
            return fn(*args, **kwargs)
        except (HTTPError, RequestException) as e:
            if attempt == max_attempts - 1:
                raise
            delay = base_delay * (2**attempt) + random.uniform(0, 1)
            print(f"  retry {attempt + 1}/{max_attempts} in {delay:.1f}s ({e})")
            time.sleep(delay)
 
 
def find_covering_items(df, buffer_meters):
    """A single STAC search whose bbox covers every point in df."""
    buffer_deg = buffer_meters / 111_000.0
    overall_bbox = [
        df["longitude"].min() - buffer_deg,
        df["latitude"].min() - buffer_deg,
        df["longitude"].max() + buffer_deg,
        df["latitude"].max() + buffer_deg,
    ]
    search = _retry(catalog.search, collections=["esa-worldcover"], bbox=overall_bbox)
    items = list(search.items())
    if not items:
        raise RuntimeError("No ESA WorldCover items found for this area.")
    print(f"Found {len(items)} WorldCover tile(s) covering all {len(df)} points.")
    return items
 
 
def item_for_point(items, lat, lon):
    """Pick whichever cached tile actually contains this point."""
    for item in items:
        w, s, e, n = item.bbox
        if w <= lon <= e and s <= lat <= n:
            return item
    # Fallback for a point sitting exactly on a tile seam: nearest tile centre
    def centre_dist(it):
        w, s, e, n = it.bbox
        return (lon - (w + e) / 2) ** 2 + (lat - (s + n) / 2) ** 2
 
    return min(items, key=centre_dist)
 
 
def get_raster(item):
    if item.id not in _raster_cache:
        _raster_cache[item.id] = _retry(
            rioxarray.open_rasterio, item.assets["map"].href, masked=True
        )
    return _raster_cache[item.id]
 
 
def get_green_coverage(item, lat, lon, buffer_meters=BUFFER_METERS):
    """Percentage of Trees(10)+Grassland(30) pixels within buffer_meters of (lat, lon)."""
    buffer_deg = buffer_meters / 111_000.0
    bbox = [lon - buffer_deg, lat - buffer_deg, lon + buffer_deg, lat + buffer_deg]
 
    rds = get_raster(item)
    rds_clipped = _retry(rds.rio.clip_box, *bbox)
    rds_projected = rds_clipped.rio.reproject("EPSG:32636")
 
    point_utm_x, point_utm_y = transformer.transform(lon, lat)
    x_coords, y_coords = rds_projected.x.values, rds_projected.y.values
    grid_x, grid_y = np.meshgrid(x_coords, y_coords)
    dist_from_center = ((grid_x - point_utm_x) ** 2 + (grid_y - point_utm_y) ** 2) ** 0.5
 
    circle_mask = dist_from_center <= buffer_meters
    raster_data = rds_projected.values[0]
    valid_pixels = raster_data[circle_mask]
 
    if len(valid_pixels) == 0:
        return 0.0
 
    green_pixels = (valid_pixels == 10) | (valid_pixels == 30)
    return round(float((green_pixels.sum() / len(valid_pixels)) * 100), 2)


In [3]:
df = pd.read_csv("../data/cairo_neighborhoods.csv")
df.head(5)

,name,name_en,latitude,longitude
0,مصر الجديدة,Heliopolis,30.117722,31.332304
1,عابدين,Abdin,30.044988,31.248863
2,بولاق,Boulak,30.059186,31.234178
3,منشية ناصر,Manshiet Naser,30.036332,31.276329
4,روض الفرج,Rod El-Farag,30.073230,31.236333


In [9]:
def main(): 
    # Resume support: skip rows already computed in a previous run.
    if os.path.exists(OUTPUT_CSV):
        done = pd.read_csv(OUTPUT_CSV)
        done_keys = set(zip(done["latitude"], done["longitude"]))
        results = done.to_dict("records")
        print(f"Resuming: {len(done)} rows already done.")
    else:
        done_keys = set()
        results = []
 
    remaining = df[
        ~df.apply(lambda r: (r["latitude"], r["longitude"]) in done_keys, axis=1)
    ]
    if remaining.empty:
        print("Nothing left to do.")
        return
 
    items = find_covering_items(df, BUFFER_METERS)
 
    for i, (_, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining))):
        item = item_for_point(items, row["latitude"], row["longitude"])
        try:
            pct = get_green_coverage(item, row["latitude"], row["longitude"])
        except Exception as e:
            print(f"  FAILED {row['name']}: {e}")
            pct = np.nan
 
        results.append(
            {
                "name": row["name"],
                "name_en": row.get("name_en"),
                "latitude": row["latitude"],
                "longitude": row["longitude"],
                "green_coverage_pct": pct,
            }
        )
 
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
 
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
    print(f"Done. Saved {len(results)} rows to {OUTPUT_CSV}")

In [10]:
main()

Found 4 WorldCover tile(s) covering all 1446 points.


100%|██████████| 1446/1446 [01:18<00:00, 18.49it/s]

Done. Saved 1446 rows to cairo_neighborhoods_green.csv


In [11]:
df_green = pd.read_csv('../data/cairo_neighborhoods_green.csv')
df_green.head()

,name,name_en,latitude,longitude,green_coverage_pct
0,مصر الجديدة,Heliopolis,30.117722,31.332304,1.52
1,عابدين,Abdin,30.044988,31.248863,0.55
2,بولاق,Boulak,30.059186,31.234178,0.56
3,منشية ناصر,Manshiet Naser,30.036332,31.276329,0.69
4,روض الفرج,Rod El-Farag,30.073230,31.236333,0.13


In [13]:
df_green['green_coverage_pct'].max()

np.float64(50.42)

In [10]:
df.to_csv("../data/greenCovarage.csv", sep=",", index=False, header=True)